# Documentation

In [1]:
from IPython.core.display import HTML

HTML(
    """<style>
    .MathJax_Display { font-size: 23px; }
</style>"""
)

$$
\theta = \theta + \alpha (r + \gamma max_{a'} Q(s',a') - Q(s,a)) \nabla_{\theta} Q(s,a)
$$

# Setup

In [1]:
"""
Add parent directorys to current path
"""

import os.path
import sys


for p in ["..", "../..", "../../..", "../../../.."]:
    d = os.path.abspath(p)
    if d not in sys.path:
        sys.path.insert(0, d)

"""
Add tiger-env directory to current path
Still not sure why this is needed.
"""
d = [
    os.path.abspath("../../../../../custom_envs/gym-tiger"),
    os.path.abspath("../../../../../custom_envs/gym-dummy/"),
]
for _d in d:
    if _d not in sys.path:
        sys.path.insert(0, _d)


"""
Enable hot-reloading
"""
from notebook_utils import import_module_by_name, reload_module_by_name


def reload():
    reload_module_by_name("research.neural_networks.mlp", "ReLU")
    reload_module_by_name("research.neural_networks.mlp", "Sigmoid")
    reload_module_by_name("research.neural_networks.mlp", "MLPRegressor")
    reload_module_by_name(
        "experiments.qlearning.dqn.dqn_seq_obs.numpy_seq_dqn", "NumpySeqDQN"
    )
    global ReLU, Sigmoid, MLPRegressor
    global NumpySeqDQN
    from research.neural_networks.mlp import ReLU, MLPRegressor
    from experiments.qlearning.dqn.dqn_seq_obs.numpy_seq_dqn import NumpySeqDQN


import gym
import matplotlib.pyplot as plt
from tabulate import tabulate

from research.neural_networks.mlp import MLPRegressor, ReLU, Sigmoid
from experiments.qlearning.dqn.dqn_seq_obs.numpy_seq_dqn import (
    NumpySeqDQN,
    play_one,
    main,
    running_avg,
    plot_running_avg,
)

# TwoInARow-v0

In [46]:
from copy import deepcopy
import gym_dummy

env = gym.make("TwoInARow-v0")
env.__init__(max_steps_per_episode=100)
copy_period = 10
gamma = 0.9
start_obs = env.reset()
obs_seq_len = 2
D = obs_seq_len * 1
K = env.action_space.n
hidden_layer_sizes = [3, 3]
hidden_layer_opts = {
    "hidden_layer_sizes": hidden_layer_sizes,
    "Z": Sigmoid(),
    "learning_rate": 1e-2,
    "mu": 0.7,
}
model = NumpySeqDQN(
    env, D, K, hidden_layer_opts, gamma, obs_seq_len, start_obs, min_experiences=4
)

N = 500
totalrewards = np.zeros(N)


print(model)

window = int(N / 10)
for n in range(N):
    if n >= (N - window):
        eps = 0
    else:
        eps = 1.0 / (n + 1) ** 0.2
    totalreward = play_one(env, model, eps, gamma, copy_period, store_seq_counts=False)
    totalrewards[n] = totalreward
    if window > 0 and n % window == 0:
        print(model)
        ravg = running_avg(totalrewards, n, window)
        print(
            "episode:",
            n,
            "total reward:",
            totalreward,
            "eps:",
            eps,
            "avg reward (last {}):".format(window),
            ravg,
        )

print(model)


print("avg reward for last {} episodes:".format(window), totalrewards[-window:].mean())

plt.plot(totalrewards)
plt.title("Rewards")
plt.show()

plot_running_avg(totalrewards, window)

# Cartpole-v0

In [75]:
from copy import deepcopy

env = gym.make("CartPole-v0")
copy_period = 10
batch_size = 32
gamma = 0.8
start_obs = env.reset()
obs_seq_len = 2
D = obs_seq_len * len(env.observation_space.sample())
K = env.action_space.n
hidden_layer_sizes = [3000, 25]
hidden_layer_opts = {
    "hidden_layer_sizes": hidden_layer_sizes,
    "Z": ReLU(),
    "learning_rate": 1e-5,
    "mu": 0.4,
}
model = NumpySeqDQN(env, D, K, hidden_layer_opts, gamma, obs_seq_len, start_obs)

N = int(1e5)
totalrewards = np.zeros(N)

window = int(N / 1000)
for n in range(N):
    eps = 1.0 / (n + 1) ** 0.3
    totalreward = play_one(env, model, eps, gamma, copy_period, store_seq_counts=False)
    totalrewards[n] = totalreward
    if n % window == 0:
        ravg = running_avg(totalrewards, n, window)
        print(
            "episode:",
            n,
            "total reward:",
            totalreward,
            "eps:",
            eps,
            "avg reward (last 10):",
            ravg,
        )
        if ravg > 150:
            break


print("avg reward for last 10 episodes:", totalrewards[-10:].mean())
print("total steps:", totalrewards.sum())

plt.plot(totalrewards)
plt.title("Rewards")
plt.show()

plot_running_avg(totalrewards, window)

# Tiger-v0

In [167]:
import gym_tiger

OBS_GROWL_LEFT = [1, 0, 0]
OBS_GROWL_RIGHT = [0, 1, 0]
OBS_START = [0, 0, 1]

ACTION_OPEN_LEFT = 0
ACTION_OPEN_RIGHT = 1
ACTION_LISTEN = 2
ACTION_MAP = {
    ACTION_OPEN_LEFT: "OPEN_LEFT",
    ACTION_OPEN_RIGHT: "OPEN_RIGHT",
    ACTION_LISTEN: "LISTEN",
}

from copy import deepcopy

env = gym.make("Tiger-v0")
env.__init__(
    reward_tiger=-100, reward_gold=10, reward_listen=-1, max_steps_per_episode=100
)
gamma = 0.99
start_obs = env.reset()

obs_seq_len = 2
hidden_layer_sizes = [15, 15]
copy_period = 10
D = obs_seq_len * env.observation_space.n
K = env.action_space.n
hidden_layer_opts = {
    "hidden_layer_sizes": hidden_layer_sizes,
    "Z": Sigmoid(),
    "learning_rate": 1e-4,
    "mu": 0.3,
}
model = NumpySeqDQN(env, D, K, hidden_layer_opts, gamma, obs_seq_len, start_obs)

N = 1000
totalrewards = np.zeros(N)

obs_perms = [
    [OBS_START, OBS_START],
    [OBS_START, OBS_GROWL_LEFT],
    [OBS_START, OBS_GROWL_RIGHT],
    [OBS_GROWL_LEFT, OBS_START],
    [OBS_GROWL_LEFT, OBS_GROWL_LEFT],
    [OBS_GROWL_LEFT, OBS_GROWL_RIGHT],
    [OBS_GROWL_RIGHT, OBS_START],
    [OBS_GROWL_RIGHT, OBS_GROWL_LEFT],
    [OBS_GROWL_RIGHT, OBS_GROWL_RIGHT],
]

window = int(N / 10)
for n in range(N):
    eps = 1.0 / np.sqrt(n + 1)
    #     eps = 1/(n+1)**(1/5)
    totalreward = play_one(env, model, eps, gamma, copy_period)
    totalrewards[n] = totalreward
    if n % window == 0:
        ravg = running_avg(totalrewards, n, window)
        print(
            "\nepisode:",
            n,
            "total reward:",
            totalreward,
            "eps:",
            eps,
            "avg reward (last 100):",
            ravg,
        )
        Q = []
        for perm in obs_perms:
            o1, o2 = perm
            _o1 = env.translate_obs(o1)
            _o2 = env.translate_obs(o2)
            pred = model.predict([o1, o2])
            best_action_idx = np.argmax(pred[0])
            action_values = pred[0].astype(str)
            action_values[best_action_idx] = action_values[best_action_idx] + " <<"
            a_openl, a_openr, a_listen = action_values
            Q.append([_o1, _o2, a_openl, a_openr, a_listen])
        print(
            "\n"
            + tabulate(
                Q,
                headers=[
                    "obs1",
                    "obs2",
                    "OPEN LEFT Q Value",
                    "OPEN RIGHT Q Value",
                    "LISTEN Q Value",
                ],
            )
        )
#         display(sorted(model.train_obs_seq_counts.items(), key=lambda x: x[1], reverse=True))
#         display(sorted(model.train_obs_seq_action_counts.items(), key=lambda x: x[1], reverse=True))


print("avg reward for last 100 episodes:", totalrewards[-10:].mean())
print("total steps:", totalrewards.sum())

In [160]:
model.train_obs_seq_counts

In [161]:
model.train_obs_seq_action_counts